In [1]:
import numpy as np
import urllib

In [2]:
graph_1 = np.array([
    [5, 8, 6, 4],
    [1, 2, 1, 2],
    [3, 2, 8, 5],
    [5, 4, 6, 3],
])

graph_2 = np.array([
    [5, 10, 15, 20, 15],
    [8, 16, 10, 11, 16],
    [4, 12, 10, 6,  5],
    [9, 3,  6,  8,  18],
    [1, 2,  8,  4,  3]
])

graph_3 = np.array([
    [32, 28, 4, 26, 4],
    [17, 19, 4, 17, 4],
    [4, 4, 5, 4, 4],
    [17, 14, 4, 14, 4],
    [21, 16, 4, 13, 4]
])

graph_4 = np.array([
    [10, 1, 10],
    [1, 10, 10],
    [10, 10, 1]
])

graph_5 = np.array([
    [5, 2, 8, 4],
    [1, 7, 3, 6],
    [9, 4, 5, 2]
])

graph_6 = np.array([
    [1, 2, 3, 4],
    [1, 2, 3, 4],
    [1, 2, 3, 4],
    [1, 2, 3, 4]
])

In [3]:
class hungarian:
    def _balanc_matrix(self, graph):
        n, m = graph.shape
        if n == m:
            return graph
        elif n > m:
            return np.hstack((graph, np.zeros((n, n-m))))
        else:
            return np.vstack((graph, np.zeros((m-n, m))))

    def _find_path(self, row, visited):
        next_nodes = np.where(self.graph[row] == 0)[0]
        for col in next_nodes:
            if col not in visited:
                visited.append(col) 
                
                if self.matched_nodes[col] == -1:
                    self.matched_nodes[col] = row
                    return True
                    
                owner_row = self.matched_nodes[col]
                
                if self._find_path(owner_row, visited):
                    self.matched_nodes[col] = row
                    return True

        return False
    
    def find(self, graph):
        self.graph = self._balanc_matrix(graph)
        n, m = self.graph.shape
        self.matched_nodes = [-1] * n

        row_mins = self.graph.min(axis=1, keepdims=True)
        self.graph = self.graph - row_mins
        cols_mins = self.graph.min(axis=0, keepdims=True)
        self.graph = self.graph - cols_mins

        for i in range(n):
            while True:
                visited = []
                if self._find_path(i, visited):
                    break
                else:
                    confilct_columns = visited
                    confilct_rows = [i]
                    for j in visited:
                        confilct_rows.append(self.matched_nodes[j])

                    col_mask = np.delete(np.arange(n), confilct_columns)
                    delta = np.min(self.graph[confilct_rows][:, col_mask])
                    self.graph[confilct_rows] = self.graph[confilct_rows] - delta 
                    self.graph[:, confilct_columns] = self.graph[:, confilct_columns] + delta
                    
        return self.matched_nodes

def export_matching_to_dot(matched_nodes, graph_name="Matching"):
    n = len(matched_nodes)
    lines = [f"digraph {graph_name} {{"]
    lines.append("    rankdir=LR;")
    lines.append("    node [shape=ellipse, width=0.5];")
    
    left_nodes = [f"L{i}" for i in range(n)]
    right_nodes_ordered = [f"R{col}" for col in matched_nodes]
    
    for i, col in enumerate(matched_nodes):
        lines.append(f'    L{i} -> R{col} [color="red", penwidth=2.0];')

    for i in range(n):
        for col in range(n):
            if col != matched_nodes[i]:
                lines.append(f'    L{i} -> R{col} [color="gray", style="dashed", constraint=false];')

    lines.append("}")
    encoded_dot = urllib.parse.quote( "\n".join(lines))

    return f"https://dreampuf.github.io/GraphvizOnline/#{encoded_dot}"

In [8]:
model = hungarian()

In [9]:
export_matching_to_dot(model.find(graph_1))

'https://dreampuf.github.io/GraphvizOnline/#digraph%20Matching%20%7B%0A%20%20%20%20rankdir%3DLR%3B%0A%20%20%20%20node%20%5Bshape%3Dellipse%2C%20width%3D0.5%5D%3B%0A%20%20%20%20L0%20-%3E%20R0%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L1%20-%3E%20R2%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L2%20-%3E%20R1%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L3%20-%3E%20R3%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L0%20-%3E%20R1%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L0%20-%3E%20R2%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L0%20-%3E%20R3%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L1%20-%3E%20R0%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L1%20-%3E%20R1%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constr

In [10]:
export_matching_to_dot(model.find(graph_2))

'https://dreampuf.github.io/GraphvizOnline/#digraph%20Matching%20%7B%0A%20%20%20%20rankdir%3DLR%3B%0A%20%20%20%20node%20%5Bshape%3Dellipse%2C%20width%3D0.5%5D%3B%0A%20%20%20%20L0%20-%3E%20R0%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L1%20-%3E%20R3%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L2%20-%3E%20R1%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L3%20-%3E%20R4%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L4%20-%3E%20R2%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L0%20-%3E%20R1%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L0%20-%3E%20R2%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L0%20-%3E%20R3%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L0%20-%3E%20R4%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%

In [11]:
export_matching_to_dot(model.find(graph_3))

'https://dreampuf.github.io/GraphvizOnline/#digraph%20Matching%20%7B%0A%20%20%20%20rankdir%3DLR%3B%0A%20%20%20%20node%20%5Bshape%3Dellipse%2C%20width%3D0.5%5D%3B%0A%20%20%20%20L0%20-%3E%20R2%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L1%20-%3E%20R3%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L2%20-%3E%20R1%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L3%20-%3E%20R4%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L4%20-%3E%20R0%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L0%20-%3E%20R0%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L0%20-%3E%20R1%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L0%20-%3E%20R3%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L0%20-%3E%20R4%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%

In [12]:
export_matching_to_dot(model.find(graph_4))

'https://dreampuf.github.io/GraphvizOnline/#digraph%20Matching%20%7B%0A%20%20%20%20rankdir%3DLR%3B%0A%20%20%20%20node%20%5Bshape%3Dellipse%2C%20width%3D0.5%5D%3B%0A%20%20%20%20L0%20-%3E%20R1%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L1%20-%3E%20R0%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L2%20-%3E%20R2%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L0%20-%3E%20R0%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L0%20-%3E%20R2%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L1%20-%3E%20R1%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L1%20-%3E%20R2%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L2%20-%3E%20R0%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L2%20-%3E%20R1%20%5Bcolor%3D%22gray%22%2C%20s

In [13]:
export_matching_to_dot(model.find(graph_5))

'https://dreampuf.github.io/GraphvizOnline/#digraph%20Matching%20%7B%0A%20%20%20%20rankdir%3DLR%3B%0A%20%20%20%20node%20%5Bshape%3Dellipse%2C%20width%3D0.5%5D%3B%0A%20%20%20%20L0%20-%3E%20R1%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L1%20-%3E%20R0%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L2%20-%3E%20R3%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L3%20-%3E%20R2%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L0%20-%3E%20R0%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L0%20-%3E%20R2%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L0%20-%3E%20R3%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L1%20-%3E%20R1%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L1%20-%3E%20R2%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constr

In [14]:
export_matching_to_dot(model.find(graph_6))

'https://dreampuf.github.io/GraphvizOnline/#digraph%20Matching%20%7B%0A%20%20%20%20rankdir%3DLR%3B%0A%20%20%20%20node%20%5Bshape%3Dellipse%2C%20width%3D0.5%5D%3B%0A%20%20%20%20L0%20-%3E%20R3%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L1%20-%3E%20R2%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L2%20-%3E%20R1%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L3%20-%3E%20R0%20%5Bcolor%3D%22red%22%2C%20penwidth%3D2.0%5D%3B%0A%20%20%20%20L0%20-%3E%20R0%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L0%20-%3E%20R1%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L0%20-%3E%20R2%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L1%20-%3E%20R0%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constraint%3Dfalse%5D%3B%0A%20%20%20%20L1%20-%3E%20R1%20%5Bcolor%3D%22gray%22%2C%20style%3D%22dashed%22%2C%20constr